In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, coalesce, when, row_number, desc, sum as spark_sum,
    concat, substring, datediff, regexp_extract
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()
if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")
print(f"env: {env}, run_date: {run_date}")

# dtransfer_Debt

Replicates on-prem stored procedure `[PLTFRM].[dtransfer].[DebtSp]`.

Produces three tables:
- `dtransfer_debtstatic` — Loan static info: agreement, party relationships, product, slave status, N4605/N3588
- `dtransfer_debtdetail` — Loan operational: settlement, restructuring, LPS, legal activity
- `dtransfer_debtfinancial` — Loan financials: balances, interest, DPD, payments, collateral

## Table Mapping

| On-Prem | Cloud | Notes |
| --- | --- | --- |
| `DWH.ast.Asset` (agreement) | `primary.exposure` (AgreementKey) | |
| `DWH.ast.Asset` (loan) | `primary.exposure` / `derivative.dexposure` | ExposureKey, ExposureId |
| `DWH.ast.Loan` | `derivative.dexposure` | IsActiveFlag, OpenDate, MaturityDate, etc. |
| `DWH.ast.LoanFinancials` | `derivative.dexposure` | Balances (OTB, OnBalance, OffBalance, DPD) |
| `DWH.ast.LoanMetrics` | `derivative.dexposure` | AssetSegment, LPS |
| `DWH.ast.LoanInterestBehaviour` | `primary.exposure` | Interest rates |
| `DWH.ast.Product` | `primary.exposure` | ProductKey, ProductCode |
| `DWH.ast.AssetXPartyRltp` | `primary.exposure` | PartyRelationshipType (per exposure-party) |
| `DWH.prt.Party` | `primary.party` | PartyId = CDI (agreed cross-platform rule; LEFT SEMI gate on CDI, no DWH PartyId in cloud). IT confirmed the alphanumeric string domain for PartyId is accepted. |
| `DWH.ast.Settlement` + relationships | `primary.settlementexposure` | Active settlements |
| `DWH.ast.Modifications_PEE` | `primary.restructuring` | Restructuring history |
| `DWH.trn.Collections` | `primary.collection` | Payments last 12 months |
| `DWH.ast.Collateral` | `derivative.dcollateral` | Covered amount |
| `DWH.act.LegalActivity` | `primary.interaction` | Last legal activity |
| `tiresias.SlaveAccounts` | `primary.entityrelationships` | Master-Slave with SubType |
| `DWH_STG.qca/qc.*` (N4605, N3588) | `landed.qca_*` tables | QC decommissioned → all in QCA now |

## Output

- `{env}-cdh-delivery.it.dtransfer_debtstatic` (~1.6M rows)
- `{env}-cdh-delivery.it.dtransfer_debtdetail` (~1.0M rows)
- `{env}-cdh-delivery.it.dtransfer_debtfinancial` (~1.0M rows)

In [0]:
# ==================== Core Source Tables ====================

# dexposure: all active exposures (one row per exposure, Primary Owner)
dexposure = (
    spark.table(f"`{env}-cdh`.derivative.dexposure")
    .filter(col("BusinessDate") == run_date_lit)
)

# exposure: all exposure-party relationships, deduplicated per ExposureKey+CDI+RelationType
# (multiple source systems load same relationship — prioritize profits > qca > others)
w_exp = Window.partitionBy("ExposureKey", "CDI", "PartyRelationshipType").orderBy(
    when(col("SourceSystem") == "profits", 1).when(col("SourceSystem") == "qca", 2).otherwise(3)
)
exposure = (
    spark.table(f"`{env}-cdh`.primary.exposure")
    .filter(col("BusinessDate") == run_date_lit)
    .withColumn("_rn", row_number().over(w_exp))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Portfolio (exclude banking/core portfolios — not in dtransfer scope)
portfolio = (
    spark.table(f"`{env}-cdh`.primary.portfolio")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & ((col("IsBankingFlag") == False) | col("IsBankingFlag").isNull())
    )
    .select("PortfolioId", "PortfolioName")
)

# Party (all-history dedup)
# Review #7: run-date scope to match dtransfer_Customer, so DebtStatic.PartyId
# (=CDI) cannot orphan against Customer.PartyId via a historical party row.
party_all = spark.table(f"`{env}-cdh`.primary.party").filter(col("BusinessDate") == run_date_lit)
w_party = Window.partitionBy("CDI").orderBy(
    when(col("SourceSystem") == "profits", 1).when(col("SourceSystem") == "qca", 2).otherwise(3),
    col("BusinessDate").desc()
)
party = (
    party_all.withColumn("_rn", row_number().over(w_party))
    .filter(col("_rn") == 1).drop("_rn")
)

# Assignment: on-prem uses Portfolio.AssignmentDate (portfolio transfer date)
# This is already available from the base SQL as PortfolioAssignmentDate

# Product (for PrdDescription)
product = (
    spark.table(f"`{env}-cdh`.primary.product")
    .filter(col("BusinessDate") == run_date_lit)
    .select(
        col("ProductKey").alias("prod_ProductKey"),
        col("ProductDescription").alias("PrdDescription"),
    )
    .distinct()
)

# Loan Details (for gd_lpucb)
loan_details = (
    spark.table(f"`{env}-cdh`.landed.profits_loan_details")
    .filter(col("BusinessDate") == run_date_lit)
    .select(
        col("account_number").alias("ld_account_number"),
        col("gd_lpucb").cast("int").alias("gd_lpucb"),
    )
)

print("Core tables loaded.")

In [0]:
# ==================== Slave Accounts (from entityrelationships) ====================
# On-prem: tiresias.SlaveAccounts → cloud: primary.entityrelationships (Master-Slave)

from pyspark.sql.functions import collect_set, concat_ws, array_join

entity_rels_slave = (
    spark.table(f"`{env}-cdh`.primary.entityrelationships")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("EntityRelationshipType") == "Master - Slave Rltnp")
        & (col("EntityType") == "Exposure")
    )
    .select(
        col("RelatedEntityKey").alias("SlaveAccountNumber"),
        concat_ws(
            "(",
            col("EntityRelationshipType"),
            coalesce(col("EntityRelationshipSubType"), lit(""))
        ).alias("reason_part"),
    )
)

# Aggregate reasons per slave account
slave_accounts = (
    entity_rels_slave
    .groupBy("SlaveAccountNumber")
    .agg(concat_ws(" | ", collect_set("reason_part")).alias("SlaveReason"))
    .withColumn("IsSlave", lit(1))
)

print(f"Slave accounts: {slave_accounts.count():,}")

In [0]:
# ==================== N4605 Status ====================
# Cases where CEXT_DEDICATED_PROPERTY_01_NVARCHAR = 'ΝΑΙ' (Law 4605 flag)

from pyspark.sql.functions import expr

qca_cases = spark.table(f"`{env}-cdh`.landed.qca_fs_cases_snap").filter(col("BusinessDate") == run_date_lit)
qca_extra = spark.table(f"`{env}-cdh`.landed.qca_fs_cases_snap_extra").filter(col("BusinessDate") == run_date_lit)
qca_dim_cases = spark.table(f"`{env}-cdh`.landed.qca_dim_cases_static").filter(col("BusinessDate") == run_date_lit)
qca_dim_cust = spark.table(f"`{env}-cdh`.landed.qca_dim_customer_static").filter(col("BusinessDate") == run_date_lit)

n4605 = (
    qca_cases
    .join(qca_extra, on="CASE_CODE", how="inner")
    .join(qca_dim_cases, on="CASE_CODE", how="inner")
    .join(qca_dim_cust, qca_cases["CUST_CODE"] == qca_dim_cust["CUST_ID"], "inner")
    .filter(col("CEXT_DEDICATED_PROPERTY_01_NVARCHAR") == "ΝΑΙ")
    .select(
        when(
            substring(col("CASE_ACCOUNT_FOR_SEARCH"), 1, 3).rlike("^[0-9]+$"),
            col("CASE_ACCOUNT_FOR_SEARCH")
        ).otherwise(substring(col("CASE_ACCOUNT_FOR_SEARCH"), 4, 99)).alias("Account"),
        qca_dim_cust["CUST_CIF"].alias("cdi"),
        lit("ΝΑΙ").alias("n4605Status"),
    )
    .distinct()
)

# ==================== N3588 Status (Bankruptcy) ====================
# Cases in bankruptcy queues (QC decommissioned → now in QCA queues)

qca_dim_queue = spark.table(f"`{env}-cdh`.landed.qca_dim_queue").filter(col("BusinessDate") == run_date_lit)

n3588 = (
    qca_cases
    .join(qca_dim_queue, qca_cases["QUE_CODE"] == qca_dim_queue["QUE_ID"], "inner")
    .join(qca_dim_cases, on="CASE_CODE", how="inner")
    .join(qca_dim_cust, qca_cases["CUST_CODE"] == qca_dim_cust["CUST_ID"], "inner")
    .filter(
        col("QUE_DESC").isin(
            "LEGAL - ΠΡΟΠΤΩΧΕΥΤΙΚΗ ΔΙΑΔΙΚΑΣΙΑ",
            "LEGAL - ΠΡΩΠΤΩΧΕΥΤΙΚΗ ΔΙΑΔΙΚΑΣΙΑ",
            "ΠΤΩΧΕΥΣΗ",
            "LEGAL - ΠΤΩΧΕΥΣΗ ΕΠΙΧΕΙΡΗΣΕΩΝ",
        )
    )
    .select(
        when(
            substring(col("CASE_ACCOUNT_FOR_SEARCH"), 1, 3).rlike("^[0-9]+$"),
            col("CASE_ACCOUNT_FOR_SEARCH")
        ).otherwise(substring(col("CASE_ACCOUNT_FOR_SEARCH"), 4, 99)).alias("Account"),
        qca_dim_cust["CUST_CIF"].alias("cdi"),
        col("QUE_DESC").alias("n3588Desc"),
    )
    .distinct()
)

print(f"N4605 accounts: {n4605.count():,}")
print(f"N3588 accounts: {n3588.count():,}")

In [0]:
# ==================== DebtStatic ====================
# Grain: one row per (ExposureKey, PartyId) — per exposure-party relationship
# exposure: all party relationships | dexposure: loan details (Primary Owner only)

# ==================== DebtStatic Base: dexposure + exposure + portfolio ====================
# Build the DebtStatic base via SQL. Row count now gated by the LEFT SEMI party
# join (drops exposures with unresolvable CDI); see the migration verification
# checklist for the current expected count rather than a hardcoded number.
debt_static_base = spark.sql(f"""
    SELECT /*+ BROADCAST(p) */
        d.ExposureKey,
        d.ExposureDerivativeID AS LoanId,
        d.AgreementKey AS dx_AgreementKey,
        d.IsActiveFlag AS ActiveAccount,
        d.Opendate AS OriginationDate,
        d.Maturitydate AS dx_MaturityDate,
        d.OriginationAmount AS dx_OriginationAmount,
        d.ProductKey AS dx_ProductKey,
        CAST(d.ProductCode AS STRING) AS ProfitsProduct,
        d.SubProductDescription AS SprdDescription,
        d.AssetSegment AS dx_AssetSegment,
        d.PurposeKTCode AS dx_PurposeKTCode,
        d.MechanismCode AS AccMechanism,
        CAST(d.BankingUnitCode AS STRING) AS FKGD_BANK_SECTOR,
        p.PortfolioId,
        CAST(p.AssignmentDate AS DATE) AS PortfolioAssignmentDate,
        e.CDI AS exp_CDI,
        e.PartyRelationshipType,
        ROW_NUMBER() OVER (
            PARTITION BY e.ExposureKey, e.CDI, e.PartyRelationshipType
            ORDER BY CASE WHEN e.SourceSystem='profits' THEN 1 WHEN e.SourceSystem='qca' THEN 2 ELSE 3 END
        ) AS _rn
    FROM `{env}-cdh`.primary.exposure e
    INNER JOIN `{env}-cdh`.derivative.dexposure d
        ON e.ExposureKey = d.ExposureKey AND d.BusinessDate = e.BusinessDate
    INNER JOIN `{env}-cdh`.primary.portfolio p
        ON p.PortfolioName = d.PortfolioName AND p.BusinessDate = d.BusinessDate
        AND (p.IsBankingFlag = false OR p.IsBankingFlag IS NULL)
    WHERE e.BusinessDate = '{run_date}'
""")
debt_static_base = debt_static_base.filter(col("_rn") == 1).drop("_rn")

# Party gate = LEFT SEMI on CDI: keeps only exposures whose customer resolves in
# party (drops NULL/unresolvable CDI), matching on-prem DebtSp (INNER JOIN
# DWH.prt.Party). SEMI (not INNER) cannot duplicate rows if party dedup changes.
debt_static_gated = debt_static_base.join(
    party.select(col("CDI").alias("pty_CDI")),
    col("exp_CDI") == col("pty_CDI"), "left_semi",
)

# Review #12: the gate silently drops rows; make the drop visible and warn when
# it is larger than expected (today ~0%). Warning only - does not block the run.
_base_cnt = debt_static_base.count()
_gated_cnt = debt_static_gated.count()
_dropped = _base_cnt - _gated_cnt
_drop_pct = (_dropped / _base_cnt) if _base_cnt else 0.0
print(f"DebtStatic party gate: {_base_cnt:,} -> {_gated_cnt:,} "
      f"(dropped {_dropped:,}, {_drop_pct:.2%})")
if _drop_pct > 0.01:
    print(f"WARNING: party gate dropped {_drop_pct:.2%} of DebtStatic rows "
          f"(> 1% threshold) - investigate upstream party/CDI coverage.")

# Continue with enrichment joins
debt_static = (
    debt_static_gated
    # PartyId = CDI (agreed cross-platform rule), taken from the exposure's own CDI.
    .withColumn("PartyId", col("exp_CDI"))
    .join(slave_accounts, col("ExposureKey") == slave_accounts["SlaveAccountNumber"], "left")
    .join(n4605, (substring(col("ExposureKey"), 4, 100) == n4605["Account"]) & (col("exp_CDI") == n4605["cdi"]), "left")
    .join(n3588, (substring(col("ExposureKey"), 4, 100) == n3588["Account"]) & (col("exp_CDI") == n3588["cdi"]), "left")
    .join(product, col("dx_ProductKey") == product["prod_ProductKey"], "left")
    .join(loan_details, col("ExposureKey") == loan_details["ld_account_number"], "left")
    .select(
        current_timestamp().alias("ETLLoadingTime"),
        run_date_lit.alias("ReferenceDate"),
        col("PortfolioId"),
        col("PartyId"),
        lit(None).cast("long").alias("AgreementId"),
        col("dx_AgreementKey").alias("AgreementNumber"),
        when(col("PartyRelationshipType") == "Primary Owner", lit("Agreement-Main Beneficiary Rltp"))
        .when(col("PartyRelationshipType") == "Guarantor", lit("Agreement-Guarantor Rltp"))
        .when(col("PartyRelationshipType") == "Co-Owner", lit("Agreement-Other Beneficiaries Rltp"))
        .alias("AgreementRelationType"),
        lit(None).cast("int").alias("AgreementRelationTypeCode"),
        col("LoanId"),
        col("ExposureKey").alias("AccountNumber"),
        col("PartyRelationshipType").alias("AccountRelationType"),
        lit(None).cast("int").alias("AccountRelationTypeCode"),
        col("ActiveAccount"),
        col("PortfolioAssignmentDate").alias("AssignmentDate"),
        col("OriginationDate").cast("date"),
        col("AccMechanism"),
        col("dx_ProductKey").alias("ProductKey"),
        col("ProfitsProduct"),
        col("PrdDescription"),
        col("SprdDescription"),
        col("dx_AssetSegment").alias("AssetSegment"),
        col("FKGD_BANK_SECTOR"),
        col("dx_MaturityDate").cast("date").alias("MaturityDate"),
        col("gd_lpucb"),
        col("dx_OriginationAmount").alias("OriginationAmount"),
        coalesce(col("IsSlave"), lit(0)).alias("IsSlave"),
        col("SlaveReason"),
        col("n4605Status"),
        col("n3588Desc").alias("n3588Status"),
        col("dx_PurposeKTCode").cast("string").alias("FKGD_CBPURP"),
    )
    .distinct()
)

print(f"DebtStatic rows: {debt_static.count():,}")
# Review #14: PartyId must equal the exposure CDI it was derived from. In
# DebtStatic, AccountNumber is also the exposure key but PartyId is the CDI, so
# assert PartyId is non-null and present on every row (the CDI is the join key).
_bad_partyid = debt_static.filter(col("PartyId").isNull()).limit(1).count()
assert _bad_partyid == 0, "DebtStatic has NULL PartyId rows (PartyId=CDI invariant violated)"

In [0]:
# ==================== DebtDetail ====================
# Grain: one row per ExposureKey (from dexposure = Primary Owner active loans)

# Settlement info (active settlements per exposure)
settlement = (
    spark.table(f"`{env}-cdh`.primary.settlementexposure")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("IsActiveFlag") == 1)
    )
    .select(
        col("ExposureKey").alias("sett_ExposureKey"),
        col("SettlementKey").cast("string").alias("settlementCode"),
        col("SettlementStatus").alias("settlementStatus"),
        col("OverdueDate").cast("date").alias("SettlementOverdueDate"),
    )
)

# Last legal activity per exposure
w_legal = Window.partitionBy("ExposureKey").orderBy(col("InteractionDate").desc())
last_legal_activity = (
    spark.table(f"`{env}-cdh`.primary.interaction")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("IsLegalFlag") == True)
    )
    .withColumn("_rn", row_number().over(w_legal))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select(
        col("ExposureKey").alias("legal_ExposureKey"),
        col("InteractionTypeKey").alias("LastLegalActivityCode"),
        col("InteractionType").alias("LastlegalActivityName"),
    )
)

# ==================== Restructuring: Step 1 — Cepal-era (drestructuring) ====================
from pyspark.sql.functions import count as spark_count
num_restructurings_cepal = (
    spark.table(f"`{env}-cdh`.derivativeactive.drestructuring")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("IsRestructureActive") == 1)
    )
    .groupBy(col("NewExposureKey").alias("rstr_key"))
    .agg(spark_count("*").alias("NumRestrCepal"))
)

# ==================== Restructuring: Step 2 — Pre-Cepal (reference table) ====================
# Reference table has historical restructurings at agreement level (pre-Cepal era).
# We use it to fill the gap for exposures NOT already flagged by drestructuring.
# Deduplicate by AgreementKey to avoid row inflation on join.
from pyspark.sql.functions import max as spark_max
modifications_pee = (
    spark.table(f"`{env}-cdh`.reference.modifications_pee_20250217")
    .filter(col("NumberofRestructurings") > 0)
    .groupBy(col("AgreementKey"))
    .agg(
        spark_max("NumberofRestructurings").alias("NumRestrPreCepal"),
        spark_max("LastRestructureDate").alias("PreCepalLastRestructureDate"),
    )
    .withColumnRenamed("AgreementKey", "mod_AgreementKey")
)

# Build DebtDetail from dexposure
debt_detail = (
    dexposure
    .join(portfolio, on="PortfolioName", how="inner")
    # Review #6: restrict to the DebtStatic LoanId set via LEFT SEMI, so
    # DebtStatic / DebtDetail / DebtFinancial share exactly one LoanId set by
    # construction (party gate is applied once, on DebtStatic). Matches on-prem.
    .join(debt_static.select(col("LoanId").alias("_ds_loanid")).distinct(),
          dexposure["ExposureDerivativeID"] == col("_ds_loanid"), "left_semi")
    .join(settlement, substring(dexposure["ExposureKey"], 4, 100) == settlement["sett_ExposureKey"], "left")
    .join(last_legal_activity, substring(dexposure["ExposureKey"], 4, 100) == last_legal_activity["legal_ExposureKey"], "left")
    .join(num_restructurings_cepal, dexposure["ExposureKey"] == num_restructurings_cepal["rstr_key"], "left")
    .join(modifications_pee, dexposure["AgreementKey"] == modifications_pee["mod_AgreementKey"], "left")
    .select(
        current_timestamp().alias("ETLLoadingTime"),
        run_date_lit.alias("ReferenceDate"),
        col("PortfolioId"),
        dexposure["ExposureDerivativeID"].alias("LoanId"),
        dexposure["DenouncedDate"].cast("date").alias("DenouncedDate"),
        dexposure["BucketCode"].cast("int").alias("SourceBucketId"),
        dexposure["ExposureStatus"].alias("LoanStatusLit"),
        dexposure["HasActiveSettlementFlag"],
        dexposure["BucketCode"].cast("int").alias("BucketId"),
        dexposure["SettlementBucketCode"].cast("int").alias("SettlementBucketId"),
        col("settlementCode"),
        col("settlementStatus"),
        when(
            dexposure["HasActiveSettlementFlag"] == True,
            datediff(run_date_lit, col("SettlementOverdueDate"))
        ).alias("SettlementDPD"),
        col("SettlementOverdueDate"),
        # IsRestructuredFlag: True if Cepal restructuring exists, or pre-Cepal exists AND not Written Off
        when(
            (col("NumRestrCepal") > 0) | ((col("NumRestrPreCepal") > 0) & (dexposure["ExposureStatus"] != "Writen Off")), lit(True)
        ).otherwise(lit(None).cast("boolean")).alias("IsRestructuredFlag"),
        # NumberOfRestructurings: sum of Cepal + pre-Cepal (pre-Cepal excluded for Written Off)
        (coalesce(col("NumRestrCepal"), lit(0)) + when(dexposure["ExposureStatus"] != "Writen Off", coalesce(col("NumRestrPreCepal"), lit(0))).otherwise(lit(0))).cast("int").alias("NumberOfRestructurings"),
        # LastRestructureDate: most recent between Cepal and pre-Cepal (pre-Cepal excluded for Written Off)
        when(
            col("NumRestrCepal") > 0, dexposure["LastRestructuringDate"].cast("date")
        ).when(
            (col("NumRestrPreCepal") > 0) & (dexposure["ExposureStatus"] != "Writen Off"),
            col("PreCepalLastRestructureDate").cast("date")
        ).alias("LastRestructureDate"),
        dexposure["LPS"],
        dexposure["LPSDate"].cast("date").alias("LPSDate"),
        col("LastLegalActivityCode"),
        col("LastlegalActivityName"),
        lit(None).cast("string").alias("n3588Code"),
        lit(None).cast("string").alias("n3588Desc"),
    )
    .distinct()
)

print(f"DebtDetail rows: {debt_detail.count():,}")

In [0]:
# ==================== Payments Last 12 Months ====================
from pyspark.sql.functions import date_sub, months_between, expr

# Use latest snapshot (BusinessDate = run_date) from primary.collection.
# Collection ExposureKey has mixed formats (with/without portfolio prefix).
# Two-pass join: direct match for prefixed, stripped match for bare-number collections.
_collections = (
    spark.table(f"`{env}-cdh`.primary.collection")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("ActualDate") >= expr(f"date_sub('{run_date}', 365)"))
        & (col("ActualDate") <= run_date_lit)
        & (col("FinalReason") == "Matched")
    )
    .select(col("ExposureKey").alias("coll_ExposureKey"), col("Amount"))
)

# Pass 1: prefixed collections → direct match to dexposure
_coll_prefixed = _collections.filter(col("coll_ExposureKey").rlike("^[A-Za-z]"))
_pay_direct = (
    _coll_prefixed
    .join(dexposure.select(col("ExposureKey").alias("dx_key")), col("coll_ExposureKey") == col("dx_key"), "inner")
    .select(col("dx_key").alias("pay_key"), "Amount")
)

# Pass 2: bare-number collections → match via stripped dexposure key
# Confirmed: bare-number collections are all from non-core portfolios
_coll_bare = _collections.filter(
    (~col("coll_ExposureKey").rlike("^[A-Za-z]"))
    & (col("coll_ExposureKey").isNotNull())
    & (col("coll_ExposureKey") != "")
)
_dx_stripped = dexposure.select(
    col("ExposureKey").alias("dx_full"),
    substring(col("ExposureKey"), 4, 100).alias("dx_bare")
).filter((col("dx_bare") != "") & col("dx_bare").isNotNull())
_pay_bare = (
    _coll_bare
    .join(_dx_stripped, col("coll_ExposureKey") == col("dx_bare"), "inner")
    .select(col("dx_full").alias("pay_key"), "Amount")
)

# Union and aggregate per active ExposureKey
payments_last_year = (
    _pay_direct.union(_pay_bare)
    .groupBy(col("pay_key").alias("pay_ExposureKey"))
    .agg(spark_sum("Amount").alias("payment_last_year"))
)

# ==================== Covered Amount (Collateral) ====================
# Sum of active collateral amount per agreement, excluding personal guarantees

personal_guarantee_exclusions = [
    "Εταιρικές Εγγυήσεις", "Προσωπική Εγγυηση Συγκ. Ποσού",
    "Προσωπική Εγγύηση", "Εγγύηση Ελληνικού Δημοσίου",
    "Προσωπική Εγγύηση Συνόλου Σύμβασης", "Ετ Εγγ Συγκ. Ποσού",
    "Εγγύηση Ελλ. Δημοσ.", "ΕΓΓΥΗΣΗ ΝΠΔΔ", "ΕΓΓΥΗΣΗ ΕΤΕΑΝ",
    "Εταιρικές Εγγυήσεις Συγκ. Ποσού",
    "Προσωπικές Εγγυήσεις (Ορισμένου ποσού)",
]

covered_amount = (
    spark.table(f"`{env}-cdh`.derivative.dcollateral")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("CollateralStatus") == "Active")
        & ~col("SubCategory").isin(*personal_guarantee_exclusions)
    )
    .groupBy(col("AgreementKey").alias("coll_AgreementKey"))
    .agg(spark_sum("CollateralAmount").alias("covered_amount"))
)

print(f"Payments last year: {payments_last_year.count():,}")
print(f"Covered amounts: {covered_amount.count():,}")

In [0]:
# ==================== DebtFinancial ====================
# Grain: one row per ExposureKey (from dexposure)
# All balance/interest columns are directly available in dexposure

debt_financial = (
    dexposure
    .join(portfolio, on="PortfolioName", how="inner")
    # Review #6: restrict to the DebtStatic LoanId set via LEFT SEMI (see DebtDetail).
    .join(debt_static.select(col("LoanId").alias("_ds_loanid")).distinct(),
          dexposure["ExposureDerivativeID"] == col("_ds_loanid"), "left_semi")
    .join(payments_last_year, dexposure["ExposureKey"] == payments_last_year["pay_ExposureKey"], "left")
    .join(covered_amount, dexposure["AgreementKey"] == covered_amount["coll_AgreementKey"], "left")
    .select(
        current_timestamp().alias("ETLLoadingTime"),
        run_date_lit.alias("ReferenceDate"),
        col("PortfolioId"),
        dexposure["ExposureDerivativeID"].alias("LoanId"),
        dexposure["Currency"],
        dexposure["ExposureStatus"].alias("LoanStatus"),
        dexposure["OutstandingTotalBalance"],
        dexposure["OnBalance"],
        dexposure["OffBalance"],
        dexposure["GBV"],
        # OverdueInterest = DueAccrued + Dueinterest + DueUnrealizedinterest + Penaltyinterest + PenaltyUnrealizedinterest
        (coalesce(dexposure["DueAccrued"], lit(0)) + coalesce(dexposure["Dueinterest"], lit(0))
         + coalesce(dexposure["DueUnrealizedinterest"], lit(0)) + coalesce(dexposure["Penaltyinterest"], lit(0))
         + coalesce(dexposure["PenaltyUnrealizedinterest"], lit(0))).alias("OverdueInterest"),
        # OverdueInterestNonUrl = DueAccrued + Dueinterest + Penaltyinterest
        (coalesce(dexposure["DueAccrued"], lit(0)) + coalesce(dexposure["Dueinterest"], lit(0))
         + coalesce(dexposure["Penaltyinterest"], lit(0))).alias("OverdueInterestNonUrl"),
        # OverdueInterestUrl = DueUnrealizedinterest + PenaltyUnrealizedinterest
        (coalesce(dexposure["DueUnrealizedinterest"], lit(0))
         + coalesce(dexposure["PenaltyUnrealizedinterest"], lit(0))).alias("OverdueInterestUrl"),
        dexposure["DaysPastDueWithMateriality"].cast("int").alias("DPD"),
        dexposure["Overduedate"].cast("date").alias("OverdueDate"),
        # ov_balance: if has settlement → DueAmount else sum of delinquent components
        when(
            dexposure["HasActiveSettlementFlag"] == True,
            coalesce(dexposure["DueAmount"], lit(0))
        ).otherwise(
            coalesce(dexposure["DueCommission"], lit(0)) + coalesce(dexposure["DueExpense"], lit(0))
            + coalesce(dexposure["Dueinterest"], lit(0)) + coalesce(dexposure["DuePrincipal"], lit(0))
            + coalesce(dexposure["DueAccrued"], lit(0)) + coalesce(dexposure["DueUnrealizedinterest"], lit(0))
            + coalesce(dexposure["Penaltyinterest"], lit(0)) + coalesce(dexposure["PenaltyUnrealizedinterest"], lit(0))
        ).alias("ov_balance"),
        # interest_amount = Interest + Accrued
        (coalesce(dexposure["interest"], lit(0)) + coalesce(dexposure["Accrued"], lit(0))).alias("interest_amount"),
        # interest_penalty = DueUnrealizedinterest + Penaltyinterest + PenaltyUnrealizedinterest + Dueinterest + DueAccrued
        (coalesce(dexposure["DueUnrealizedinterest"], lit(0)) + coalesce(dexposure["Penaltyinterest"], lit(0))
         + coalesce(dexposure["PenaltyUnrealizedinterest"], lit(0)) + coalesce(dexposure["Dueinterest"], lit(0))
         + coalesce(dexposure["DueAccrued"], lit(0))).alias("interest_penalty"),
        # total_amount_secured: min(covered_amount, OutstandingTotalBalance)
        when(
            col("covered_amount").isNotNull(),
            when(col("covered_amount") > dexposure["OutstandingTotalBalance"], dexposure["OutstandingTotalBalance"])
            .otherwise(col("covered_amount"))
        ).otherwise(lit(0)).alias("total_amount_secured"),
        lit(1).cast("decimal(19,6)").alias("ExchangeRate"),  # TODO: from exchange rates table if non-EUR
        # Interest behaviour
        when(
            (dexposure["InterestRateType"] == "Fixed") & (dexposure["InterestDuration"] != 999),
            lit("Fixed Periodic")
        ).otherwise(dexposure["InterestRateType"]).alias("interest_type"),
        dexposure["BaseRate"].alias("interest_rate"),
        dexposure["SpreadRate"].alias("interest_spread"),
        dexposure["N128Rate"].alias("interest_n128"),
        coalesce(col("payment_last_year"), lit(0)).alias("payment_last_year"),
        dexposure["NextInstallmentDate"].cast("date").alias("NextInstallmentDate"),
        dexposure["PenaltyRate"],
        dexposure["interestDescription"].alias("InterestDescription"),
        dexposure["BaseDescription"],
        # FixInterestEndDate: only for Fixed Periodic
        when(
            (dexposure["InterestRateType"] == "Fixed") & (dexposure["InterestDuration"] != 999),
            dexposure["FixInterestEndDate"]
        ).alias("FixInterestEndDate"),
        # Duration: concat duration + frequency indicator
        when(
            (dexposure["InterestRateType"] == "Variable") | ((dexposure["InterestRateType"] == "Fixed") & (dexposure["InterestDuration"] == 999)),
            lit(None)
        ).otherwise(
            when(dexposure["InterestDuration"].isNull(), lit(None))
            .otherwise(concat(dexposure["InterestDuration"].cast("string"), lit("M")))
        ).alias("Duration"),
        # Detailed balance components
        dexposure["Principal"],
        dexposure["interest"].alias("Interest"),
        dexposure["Unrealizedinterest"].alias("UnrealizedInterest"),
        dexposure["Expense"],
        dexposure["Commission"].alias("Comission"),
        dexposure["Accrued"],
        dexposure["DuePrincipal"].alias("DelinquentPrincipal"),
        dexposure["Dueinterest"].alias("DelinquentInterest"),
        dexposure["DueUnrealizedinterest"].alias("DelinquentUnrInterest"),
        dexposure["Penaltyinterest"].alias("PenaltyInterest"),
        dexposure["PenaltyUnrealizedinterest"].alias("PenaltyUnrInterest"),
        dexposure["DueExpense"].alias("DelinquentExpense"),
        dexposure["DueCommission"].alias("DelinquentComission"),
        dexposure["DueAccrued"].alias("DelinquentAccrued"),
        lit(None).cast("date").alias("CutOffOverdueDate"),
    )
    .distinct()
)

print(f"DebtFinancial rows: {debt_financial.count():,}")

In [0]:
# ==================== Review #14: cross-table LoanId-set parity ====================
# On-prem guarantees DebtStatic / DebtDetail / DebtFinancial share one LoanId set
# (zero orphans). The cloud party gate is applied only to DebtStatic, so assert the
# three LoanId sets match before writing; a mismatch means a downstream drift.
_ls_static = {r[0] for r in debt_static.select("LoanId").distinct().collect()}
_ls_detail = {r[0] for r in debt_detail.select("LoanId").distinct().collect()}
_ls_fin = {r[0] for r in debt_financial.select("LoanId").distinct().collect()}
assert _ls_static == _ls_detail == _ls_fin, (
    f"LoanId sets differ: static={len(_ls_static):,}, detail={len(_ls_detail):,}, "
    f"financial={len(_ls_fin):,}; only_in_static(detail)={len(_ls_static-_ls_detail):,}, "
    f"only_in_detail={len(_ls_detail-_ls_static):,}"
)
print(f"LoanId-set parity OK: {len(_ls_static):,} loans across all three tables")

# ==================== Write DebtStatic ====================
target_static = f"`{env}-cdh-delivery`.it.dtransfer_debtstatic"
debt_static.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_static)
print(f"Written to {target_static}")

# ==================== Write DebtDetail ====================
target_detail = f"`{env}-cdh-delivery`.it.dtransfer_debtdetail"
debt_detail.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_detail)
print(f"Written to {target_detail}")

# ==================== Write DebtFinancial ====================
target_financial = f"`{env}-cdh-delivery`.it.dtransfer_debtfinancial"
debt_financial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_financial)
print(f"Written to {target_financial}")